# JurisData Analytics — Fase 2: Modelagem Relacional

**Objetivo:** Construir o modelo analítico (star schema), views de negócio e feature store para ML.
**Input:** analytics.processos (40.232 registros)
**Output:** Dimensões, tabela fato, views analíticas, ml.features_decisao

In [1]:
import os
import pandas as pd
import psycopg2
from dotenv import load_dotenv

load_dotenv()

PG_CONFIG = {
    "host":     os.getenv("PG_HOST", "localhost"),
    "port":     int(os.getenv("PG_PORT", "5432")),
    "dbname":   os.getenv("PG_DBNAME", "jurisdata"),
    "user":     os.getenv("PG_USER", "postgres"),
    "password": os.getenv("PG_PASSWORD", ""),
}

def get_conn():
    return psycopg2.connect(**PG_CONFIG)

with get_conn() as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT COUNT(*) FROM analytics.processos")
        print("Registros disponíveis:", cur.fetchone()[0])

Registros disponíveis: 40232


In [6]:
DDL_DIM_ASSUNTO = """
CREATE TABLE IF NOT EXISTS analytics.dim_assunto (
    assunto_codigo  INTEGER PRIMARY KEY,
    assunto_nome    TEXT NOT NULL,
    created_at      TIMESTAMPTZ DEFAULT NOW()
);
"""

POPULATE_DIM_ASSUNTO = """
INSERT INTO analytics.dim_assunto (assunto_codigo, assunto_nome)
SELECT DISTINCT
    assunto_codigo,
    COALESCE(assunto_principal, 'Não informado')
FROM analytics.processos
WHERE assunto_codigo IS NOT NULL
ON CONFLICT (assunto_codigo) DO NOTHING;
"""

with get_conn() as conn:
    with conn.cursor() as cur:
        cur.execute(DDL_DIM_ASSUNTO)
        cur.execute(POPULATE_DIM_ASSUNTO)
        cur.execute("SELECT COUNT(*) FROM analytics.dim_assunto")
        print("Assuntos distintos:", cur.fetchone()[0])
    conn.commit()

Assuntos distintos: 492


In [3]:
DDL_DIM_ORGAO = """
CREATE TABLE IF NOT EXISTS analytics.dim_orgao_julgador (
    id              BIGSERIAL PRIMARY KEY,
    orgao_nome      TEXT UNIQUE NOT NULL,
    created_at      TIMESTAMPTZ DEFAULT NOW()
);
"""

POPULATE_DIM_ORGAO = """
INSERT INTO analytics.dim_orgao_julgador (orgao_nome)
SELECT DISTINCT orgao_julgador
FROM analytics.processos
WHERE orgao_julgador IS NOT NULL
ON CONFLICT (orgao_nome) DO NOTHING;
"""

with get_conn() as conn:
    with conn.cursor() as cur:
        cur.execute(DDL_DIM_ORGAO)
        cur.execute(POPULATE_DIM_ORGAO)
        cur.execute("SELECT COUNT(*) FROM analytics.dim_orgao_julgador")
        print("Órgãos julgadores distintos:", cur.fetchone()[0])
    conn.commit()

Órgãos julgadores distintos: 34


In [4]:
DDL_DIM_TEMPO = """
CREATE TABLE IF NOT EXISTS analytics.dim_tempo (
    data_key    DATE PRIMARY KEY,
    ano         INTEGER NOT NULL,
    semestre    INTEGER NOT NULL,
    trimestre   INTEGER NOT NULL,
    mes         INTEGER NOT NULL,
    mes_nome    TEXT NOT NULL,
    semana      INTEGER NOT NULL
);
"""

POPULATE_DIM_TEMPO = """
INSERT INTO analytics.dim_tempo (data_key, ano, semestre, trimestre, mes, mes_nome, semana)
SELECT
    d::DATE,
    EXTRACT(YEAR FROM d)::INTEGER,
    CASE WHEN EXTRACT(MONTH FROM d) <= 6 THEN 1 ELSE 2 END,
    EXTRACT(QUARTER FROM d)::INTEGER,
    EXTRACT(MONTH FROM d)::INTEGER,
    TO_CHAR(d, 'TMMonth'),
    EXTRACT(WEEK FROM d)::INTEGER
FROM generate_series('2018-01-01'::DATE, '2024-12-31'::DATE, '1 day') d
ON CONFLICT (data_key) DO NOTHING;
"""

with get_conn() as conn:
    with conn.cursor() as cur:
        cur.execute(DDL_DIM_TEMPO)
        cur.execute(POPULATE_DIM_TEMPO)
        cur.execute("SELECT COUNT(*) FROM analytics.dim_tempo")
        print("Dias na dimensão de tempo:", cur.fetchone()[0])
    conn.commit()

Dias na dimensão de tempo: 2557


In [7]:
DDL_FATO = """
CREATE TABLE IF NOT EXISTS analytics.fato_processo (
    id                      BIGSERIAL PRIMARY KEY,
    process_id              TEXT UNIQUE NOT NULL REFERENCES analytics.processos(process_id),
    -- Chaves das dimensões
    data_ajuizamento_key    DATE REFERENCES analytics.dim_tempo(data_key),
    assunto_codigo          INTEGER REFERENCES analytics.dim_assunto(assunto_codigo),
    orgao_julgador_id       BIGINT REFERENCES analytics.dim_orgao_julgador(id),
    -- Atributos descritivos
    grau                    TEXT,
    formato                 TEXT,
    -- Métricas
    tempo_tramitacao_dias   INTEGER,
    created_at              TIMESTAMPTZ DEFAULT NOW()
);

CREATE INDEX IF NOT EXISTS idx_fato_data    ON analytics.fato_processo (data_ajuizamento_key);
CREATE INDEX IF NOT EXISTS idx_fato_assunto ON analytics.fato_processo (assunto_codigo);
CREATE INDEX IF NOT EXISTS idx_fato_orgao   ON analytics.fato_processo (orgao_julgador_id);
"""

POPULATE_FATO = """
INSERT INTO analytics.fato_processo (
    process_id,
    data_ajuizamento_key,
    assunto_codigo,
    orgao_julgador_id,
    grau,
    formato,
    tempo_tramitacao_dias
)
SELECT
    p.process_id,
    p.data_ajuizamento,
    p.assunto_codigo,
    o.id,
    p.grau,
    p.formato,
    p.tempo_tramitacao_dias
FROM analytics.processos p
LEFT JOIN analytics.dim_orgao_julgador o ON o.orgao_nome = p.orgao_julgador
ON CONFLICT (process_id) DO NOTHING;
"""

with get_conn() as conn:
    with conn.cursor() as cur:
        cur.execute(DDL_FATO)
        cur.execute(POPULATE_FATO)
        cur.execute("SELECT COUNT(*) FROM analytics.fato_processo")
        print("Registros na tabela fato:", cur.fetchone()[0])
    conn.commit()

Registros na tabela fato: 40232


In [8]:
VIEWS = """
-- Volume e tempo médio por assunto
CREATE OR REPLACE VIEW analytics.vw_assuntos AS
SELECT
    a.assunto_codigo,
    a.assunto_nome,
    COUNT(f.id)                            AS total_processos,
    ROUND(AVG(f.tempo_tramitacao_dias), 0) AS tempo_medio_dias,
    ROUND(100.0 * COUNT(f.id) / SUM(COUNT(f.id)) OVER (), 2) AS pct_total
FROM analytics.fato_processo f
JOIN analytics.dim_assunto a ON a.assunto_codigo = f.assunto_codigo
GROUP BY a.assunto_codigo, a.assunto_nome
ORDER BY total_processos DESC;

-- Volume por ano e trimestre
CREATE OR REPLACE VIEW analytics.vw_serie_temporal AS
SELECT
    t.ano,
    t.trimestre,
    t.semestre,
    COUNT(f.id)                            AS total_ajuizados,
    ROUND(AVG(f.tempo_tramitacao_dias), 0) AS tempo_medio_dias
FROM analytics.fato_processo f
JOIN analytics.dim_tempo t ON t.data_key = f.data_ajuizamento_key
GROUP BY t.ano, t.trimestre, t.semestre
ORDER BY t.ano, t.trimestre;

-- Ranking de órgãos julgadores
CREATE OR REPLACE VIEW analytics.vw_orgaos AS
SELECT
    o.orgao_nome,
    COUNT(f.id)                            AS total_processos,
    ROUND(AVG(f.tempo_tramitacao_dias), 0) AS tempo_medio_dias,
    PERCENTILE_CONT(0.5) WITHIN GROUP
        (ORDER BY f.tempo_tramitacao_dias)::INTEGER AS mediana_dias
FROM analytics.fato_processo f
JOIN analytics.dim_orgao_julgador o ON o.id = f.orgao_julgador_id
GROUP BY o.orgao_nome
ORDER BY total_processos DESC;
"""

with get_conn() as conn:
    with conn.cursor() as cur:
        cur.execute(VIEWS)
    conn.commit()
    print("Views criadas: vw_assuntos, vw_serie_temporal, vw_orgaos")

Views criadas: vw_assuntos, vw_serie_temporal, vw_orgaos


In [9]:
DDL_FEATURES = """
CREATE TABLE IF NOT EXISTS ml.features_decisao (
    id                    BIGSERIAL PRIMARY KEY,
    process_id            TEXT UNIQUE NOT NULL REFERENCES analytics.processos(process_id),
    -- Features numéricas brutas
    tempo_tramitacao_dias INTEGER,
    ano_ajuizamento       INTEGER,
    mes_ajuizamento       INTEGER,
    -- Features categóricas encoded
    assunto_codigo        INTEGER,
    grau_encoded          SMALLINT,  -- 0=G1, 1=G2, 2=JE
    formato_encoded       SMALLINT,  -- 0=Eletrônico, 1=Físico
    -- Outputs de ML (preenchidos na Fase 3)
    cluster_id            INTEGER,
    cluster_label         TEXT,
    risk_score            NUMERIC(5,2),
    embedding_x           NUMERIC(10,6),
    embedding_y           NUMERIC(10,6),
    created_at            TIMESTAMPTZ DEFAULT NOW()
);
"""

POPULATE_FEATURES = """
INSERT INTO ml.features_decisao (
    process_id,
    tempo_tramitacao_dias,
    ano_ajuizamento,
    mes_ajuizamento,
    assunto_codigo,
    grau_encoded,
    formato_encoded
)
SELECT
    p.process_id,
    p.tempo_tramitacao_dias,
    EXTRACT(YEAR FROM p.data_ajuizamento)::INTEGER,
    EXTRACT(MONTH FROM p.data_ajuizamento)::INTEGER,
    p.assunto_codigo,
    CASE p.grau
        WHEN 'G1' THEN 0
        WHEN 'G2' THEN 1
        WHEN 'JE' THEN 2
        ELSE 3
    END,
    CASE p.formato
        WHEN 'Eletrônico' THEN 0
        ELSE 1
    END
FROM analytics.processos p
WHERE p.tempo_tramitacao_dias IS NOT NULL
  AND p.assunto_codigo IS NOT NULL
ON CONFLICT (process_id) DO NOTHING;
"""

with get_conn() as conn:
    with conn.cursor() as cur:
        cur.execute(DDL_FEATURES)
        cur.execute(POPULATE_FEATURES)
        cur.execute("SELECT COUNT(*) FROM ml.features_decisao")
        print("Registros na feature store:", cur.fetchone()[0])
    conn.commit()

Registros na feature store: 40154


In [10]:
queries = {
    "dim_assunto":          "SELECT COUNT(*) FROM analytics.dim_assunto",
    "dim_orgao_julgador":   "SELECT COUNT(*) FROM analytics.dim_orgao_julgador",
    "dim_tempo":            "SELECT COUNT(*) FROM analytics.dim_tempo",
    "fato_processo":        "SELECT COUNT(*) FROM analytics.fato_processo",
    "ml.features_decisao":  "SELECT COUNT(*) FROM ml.features_decisao",
}

with get_conn() as conn:
    for tabela, sql in queries.items():
        with conn.cursor() as cur:
            cur.execute(sql)
            print(f"{tabela:30s} → {cur.fetchone()[0]:,} registros")

print("\nTop 5 assuntos (view):")
with get_conn() as conn:
    print(pd.read_sql(
        "SELECT assunto_nome, total_processos, tempo_medio_dias FROM analytics.vw_assuntos LIMIT 5",
        conn
    ).to_string(index=False))

dim_assunto                    → 492 registros
dim_orgao_julgador             → 34 registros
dim_tempo                      → 2,557 registros
fato_processo                  → 40,232 registros
ml.features_decisao            → 40,154 registros

Top 5 assuntos (view):
                        assunto_nome  total_processos  tempo_medio_dias
                  Verbas Rescisórias             4956            1375.0
                   Rescisão Indireta             2347            1310.0
                        Aviso Prévio             2146            2083.0
          Adicional de Insalubridade             1959            1300.0
Reconhecimento de Relação de Emprego             1661            1217.0


C:\Users\fabri\AppData\Local\Temp\ipykernel_8728\630980144.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  print(pd.read_sql(
